In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import OrderedDict

In [23]:
class ActionEmbedder(nn.Module):
    def __init__(self, action_size, hidden_dim):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(action_size,hidden_dim),
            nn.SiLU()
        )
    def forward(self, x):
        return self.embed(x)

In [35]:
class Attention(nn.Module):
    def __init__(self, hidden_dim, nb_heads, p_dropout):
        super().__init__()
        self.attention = nn.MultiheadAttention(hidden_dim, nb_heads, batch_first=True, dropout=p_dropout)

    def forward(self, x):
        return self.attention(x,x,x, need_weights=False, is_causal=True, attn_mask=nn.modules.transformer.Transformer.generate_square_subsequent_mask(x.shape[-2]))[0]

In [40]:
class Predictor(nn.Module):
    def __init__(self, hidden_dim, nb_layers, p_dropout, nb_heads):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.condition = nn.Linear(hidden_dim, 6*hidden_dim)
        with torch.no_grad():
            for p in self.condition.parameters():
                p.zero_()
        self.layers = nn.ModuleList([
             nn.ModuleDict({
                'norm1': nn.LayerNorm(hidden_dim,elementwise_affine=False),
                'att': Attention(hidden_dim, nb_heads, p_dropout),
                'norm2': nn.LayerNorm(hidden_dim,elementwise_affine=False),
                'ffn': nn.Sequential(nn.Linear(hidden_dim, 4*hidden_dim), nn.GELU(), nn.Linear(4*hidden_dim, hidden_dim)),
             }) for _ in range(nb_layers)
        ])
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(p_dropout)

    def forward(self, x, action):
        # x.shape = B * T * H
        # action.shape = B * T * H
        alpha1,beta1,gamma1,alpha2,beta2,gamma2 = self.condition(action).chunk(6,-1)
        for layer in self.layers:
            x = self.dropout(layer['att'](layer['norm1'](x)*(1.0+gamma1)+beta1))*alpha1+x
            x = self.dropout(layer['ffn'](layer['norm2'](x)*(1.0+gamma2)+beta2))*alpha2+x
        return self.layer_norm(x)
        

In [37]:
pred = Predictor(hidden_dim=192, nb_layers=6, nb_heads=16, p_dropout=0.1)

In [38]:
action_embedder = ActionEmbedder(4, 192)

In [39]:
pred(torch.rand(10, 100, 192), action_embedder(torch.rand(10, 100, 4)))

tensor([[[ 0.7809, -0.4569, -0.4837,  ..., -1.3123,  0.8294,  0.4205],
         [ 0.8178, -1.0644,  0.6486,  ...,  0.1940,  0.8242,  1.8261],
         [-0.6152, -0.8366,  0.6633,  ...,  1.3909,  1.0879,  0.2756],
         ...,
         [-0.3890,  0.6395,  1.7097,  ...,  0.1300,  0.9099,  1.6378],
         [-1.6300, -0.3037, -0.7164,  ...,  0.1930, -0.3296, -0.9661],
         [ 0.0060, -0.6755, -0.9253,  ..., -1.6458, -1.4904, -0.1124]],

        [[ 0.3675, -0.8214, -0.3915,  ..., -1.4657, -0.1909,  0.8977],
         [-0.9404, -1.2559, -1.1482,  ...,  1.0425, -0.0508, -0.3610],
         [-0.5098,  1.1887, -1.5502,  ..., -0.1284,  0.1378,  0.6220],
         ...,
         [ 0.3181,  1.6528, -0.4445,  ..., -1.1272,  0.7555, -1.0339],
         [ 0.9898, -0.3122, -1.0002,  ...,  0.5061,  1.1904,  0.7355],
         [ 0.3311, -1.2024,  1.3332,  ...,  0.1928, -1.0992, -0.5349]],

        [[ 0.7469, -0.6749,  0.8079,  ...,  0.4440,  1.6922, -1.4259],
         [-0.1467, -0.9921,  0.7319,  ...,  0